# Fine-Tuning a Protein Language Model for Subcellular Localization

In this lab, we'll compare two ways to adapt a protein language model, a specific type of neural network, for protein sequence classification.
After a model is pre-trained on a broad task, we can adapt it to a new task with additional training. This process is called **fine-tuning**.

Here, we'll fine-tune **AMPLIFY_120M** to predict **subcellular localization** from amino-acid sequence.

We will compare two strategies:

1. **Head-only fine-tuning**: freeze the AMPLIFY backbone and train only the final classification head.
2. **LoRA (PEFT)**: keep the backbone mostly frozen, but add small trainable adapters to selected attention layers.

The dataset is `proteinglm/localization_prediction`, which has 10 localization classes.
We evaluate both approaches with:

- **Accuracy**: overall fraction correct.
- **Macro-F1**: averages class-wise F1 equally, so minority classes matter more.

## Runtime note

This notebook is designed for **Google Colab with a GPU** (T4/L4 recommended).
Use `Runtime -> Change runtime type -> GPU` before running.
Default settings use small subsets so the notebook can finish quickly.

The next cell installs required packages for this session.


In [ ]:
%%capture
!pip install -q transformers datasets accelerate evaluate peft scikit-learn

## Import libraries and check the environment

We'll import all libraries up front and verify GPU availability.

If you see an error, you likely need to take the steps described in the "Runtime note" section above.

In [ ]:
import os
import random
import time
from collections import Counter

import evaluate
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from peft import LoraConfig, TaskType, get_peft_model
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
)

cuda_ok = torch.cuda.is_available()
device = torch.device("cuda")
if not cuda_ok:
    raise RuntimeError("Make sure you selected a GPU as mentioned above!")

## Reproducibility and configuration

All experiment settings live in one `CONFIG` dictionary.
This makes it easy to run controlled comparisons and easy for students to modify one variable at a time.

Why these defaults?
- small subset sizes (`train_n`, `eval_n`) keep runtime short,
- `max_length=256` limits memory use,
- separate learning rates for head-only and LoRA give each method a fair starting point.

We also set random seeds for Python, NumPy, and PyTorch so runs are as reproducible as possible.


In [ ]:
CONFIG = {
    "model_name": "chandar-lab/AMPLIFY_120M",  # The model that we'll be using
    # The dataset that we'll use
    "dataset_name": "proteinglm/localization_prediction",
    "train_n": 2000,  # Limit the number of training examples
    "eval_n": 500,  # Limit the number of evaluation examples
    "max_length": 256,  # Keep proteins short for speed.
    "epochs": 2,  # How many times to iterate over the data
    "lr_head": 2e-3,  # The learning rate for the classification head
    "lr_lora": 2e-4,  # The learning rate for LoRA
    "train_batch_size": 8,  # How many proteins in each training batch
    "eval_batch_size": 16,  # How many proteins in each evaluation batch
    "seed": 0,  # A random seed for reproducibility
    "lora_r": 8,  # The rank for LoRA
    "lora_alpha": 16,  # The strength of LoRA
    "lora_dropout": 0.05,  # A regularization method for LoRA
}


def set_all_seeds(seed: int):
    """Set all of the random seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ["PYTHONHASHSEED"] = str(seed)


set_all_seeds(CONFIG["seed"])
print("Seed set to", CONFIG["seed"])

## Load the dataset, subsample, and tokenize

We'll load the ProteinGLM localization dataset, shuffle with a fixed seed, and select a small subset.
Then we tokenize sequences from the `seq` field.

Important preprocessing choices:
- `truncation=True`: long sequences are cut to `max_length`,
- `padding='max_length'`: all examples have the same length for batching,
- fixed `max_length`: improves throughput and predictability on Colab GPUs.

We also print class counts so you can see whether labels are imbalanced.


In [ ]:
raw = load_dataset(CONFIG["dataset_name"])
print(raw)

if "validation" in raw:
    eval_split_name = "validation"
elif "test" in raw:
    eval_split_name = "test"

train_full = raw["train"].shuffle(seed=CONFIG["seed"])
eval_full = raw[eval_split_name].shuffle(seed=CONFIG["seed"])

train_n = min(CONFIG["train_n"], len(train_full))
eval_n = min(CONFIG["eval_n"], len(eval_full))

train_ds = train_full.select(range(train_n))
eval_ds = eval_full.select(range(eval_n))

print(f"Using train split size: {len(train_ds)}")
print(f"Using eval split size: {len(eval_ds)} from '{eval_split_name}'")

label_counts = Counter(train_ds["label"])
print("Label counts in train subset:")
for label_id, count in sorted(label_counts.items()):
    print(f"  class {label_id}: {count}")

sample_for_length = min(500, len(train_ds))
mean_len = float(
    np.mean([len(s) for s in train_ds.select(range(sample_for_length))["seq"]])
)
print(
    f"Average sequence length (sample of {sample_for_length}): {mean_len:.1f}"
)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    CONFIG["model_name"], trust_remote_code=True
)


def tokenize_batch(batch):
    """Transform a batch of protein sequences into numeric tokens."""
    return tokenizer(
        batch["seq"],
        truncation=True,
        padding="max_length",
        max_length=CONFIG["max_length"],
    )


train_tok = train_ds.map(tokenize_batch, batched=True)
eval_tok = eval_ds.map(tokenize_batch, batched=True)

keep_cols = ["input_ids", "attention_mask", "label"]
train_tok = train_tok.remove_columns(
    [c for c in train_tok.column_names if c not in keep_cols]
)
eval_tok = eval_tok.remove_columns(
    [c for c in eval_tok.column_names if c not in keep_cols]
)

train_tok = train_tok.rename_column("label", "labels")
eval_tok = eval_tok.rename_column("label", "labels")

train_tok.set_format(
    type="torch", columns=["input_ids", "attention_mask", "labels"]
)
eval_tok.set_format(
    type="torch", columns=["input_ids", "attention_mask", "labels"]
)

print("Tokenized columns (train):", train_tok.column_names)
print("Tokenized columns (eval):", eval_tok.column_names)

## Metrics helpers

We'll compute two complementary metrics:

- **Accuracy** for overall correctness.
- **Macro-F1** to give each class equal weight.

In localization tasks, class imbalance is common, so macro-F1 is often more informative than accuracy alone.
We'll also define a confusion matrix plot to inspect where errors happen.


In [ ]:
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")


def compute_metrics(eval_pred):
    """Compute accuracy and macro-F1 for evaluation."""
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=preds, references=labels)[
        "accuracy"
    ]
    macro_f1 = f1_metric.compute(
        predictions=preds,
        references=labels,
        average="macro",
    )["f1"]
    return {"accuracy": acc, "macro_f1": macro_f1}


def plot_confusion_matrix(y_true, y_pred, title, label_names=None):
    """Plot a confusion matrix for model predictions."""
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(6, 6))
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm, display_labels=label_names
    )
    disp.plot(ax=ax, xticks_rotation=45, colorbar=False)
    ax.set_title(title)
    plt.tight_layout()
    plt.show()

## Build models: head-only and LoRA

Both methods start from the same pretrained AMPLIFY checkpoint.
The key difference is **which parameters are trainable**.

- In **head-only**, only classifier parameters are updated.
- In **LoRA**, we inject trainable low-rank adapters into selected attention projections.

We print trainable parameter counts for transparency.
This is important: PEFT methods are mainly about reducing trainable parameters while keeping strong performance.


In [ ]:
NUM_LABELS = 10


def count_parameters(model):
    """Count the number of parameters in the model."""
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return trainable, total


def print_trainable_summary(model, name):
    """Print the number of parameters that will be trained."""
    trainable, total = count_parameters(model)
    pct = 100 * trainable / total
    print(f"{name} trainable params: {trainable:,} / {total:,} ({pct:.2f}%)")


def build_head_only_model():
    """Build a sequence classifier with only the head trainable."""
    model = AutoModelForSequenceClassification.from_pretrained(
        CONFIG["model_name"],
        num_labels=NUM_LABELS,
        trust_remote_code=True,
    )

    for _, param in model.named_parameters():
        param.requires_grad = False

    head_keywords = ["classifier", "score", "classification_head"]
    matched = []
    for name, param in model.named_parameters():
        if any(k in name.lower() for k in head_keywords):
            param.requires_grad = True
            matched.append(name)

    if len(matched) == 0:
        raise ValueError(
            "Could not find classification head parameters to unfreeze. "
            "Inspect model.named_parameters() for head names."
        )

    print("Head-only trainable parameter names (sample):")
    for name in matched[:10]:
        print("  -", name)
    print_trainable_summary(model, "Head-only")
    return model


def find_lora_targets(model):
    """Infer likely attention modules for LoRA injection."""
    linear_names = [
        n for n, m in model.named_modules() if isinstance(m, torch.nn.Linear)
    ]

    exact = ["q_proj", "k_proj", "v_proj", "o_proj"]
    found_exact = []
    for key in exact:
        if any(n == key or n.endswith(f".{key}") for n in linear_names):
            found_exact.append(key)
    if found_exact:
        print("LoRA target modules (exact match):", found_exact)
        return found_exact

    fallback_checks = {
        "query": ["query"],
        "key": ["key"],
        "value": ["value"],
        "out": ["out", "output", "dense"],
    }

    found_fallback = []
    for name in linear_names:
        tail = name.split(".")[-1]
        lower = name.lower()
        if "attn" in lower or "attention" in lower:
            if any(
                any(tok in lower for tok in toks)
                for toks in fallback_checks.values()
            ):
                found_fallback.append(tail)

    found_fallback = sorted(set(found_fallback))
    if found_fallback:
        print("LoRA target modules (fallback match):", found_fallback)
        return found_fallback

    print("Could not find obvious attention projection names.")
    print("First 30 linear module names for debugging:")
    for name in linear_names[:30]:
        print("  -", name)
    raise ValueError("No LoRA target modules found automatically.")


def build_lora_model():
    """Build a PEFT LoRA sequence-classification model."""
    model = AutoModelForSequenceClassification.from_pretrained(
        CONFIG["model_name"],
        num_labels=NUM_LABELS,
        trust_remote_code=True,
    )
    targets = find_lora_targets(model)

    lora_cfg = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        r=CONFIG["lora_r"],
        lora_alpha=CONFIG["lora_alpha"],
        lora_dropout=CONFIG["lora_dropout"],
        bias="none",
        target_modules=targets,
    )
    model = get_peft_model(model, lora_cfg)
    model.print_trainable_parameters()
    print_trainable_summary(model, "LoRA")
    return model

## Training runner

This helper function wraps the Hugging Face `Trainer` setup so both experiments use the same training/evaluation pipeline.

It returns:
- training output,
- evaluation metrics,
- predicted labels and true labels,
- wall-clock runtime.

Using one shared runner keeps the comparison fair and easier to explain.


In [ ]:
def _training_args(run_name, lr, batch_train, batch_eval, epochs):
    args = {
        "output_dir": f"out/{run_name}",
        "per_device_train_batch_size": batch_train,
        "per_device_eval_batch_size": batch_eval,
        "learning_rate": lr,
        "num_train_epochs": epochs,
        "logging_steps": 25,
        "save_strategy": "no",
        "report_to": "none",
        "seed": CONFIG["seed"],
        "data_seed": CONFIG["seed"],
        "fp16": cuda_ok,
    }

    # Transformers renamed this argument in newer versions.
    if "eval_strategy" in TrainingArguments.__init__.__code__.co_varnames:
        args["eval_strategy"] = "epoch"
    else:
        args["evaluation_strategy"] = "epoch"

    return TrainingArguments(**args)


def train_and_evaluate(model, run_name, lr, batch_train, batch_eval, epochs):
    """Train one model configuration and return metrics and predictions."""
    args = _training_args(run_name, lr, batch_train, batch_eval, epochs)

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_tok,
        eval_dataset=eval_tok,
        tokenizer=tokenizer,
        compute_metrics=compute_metrics,
    )

    start = time.time()
    train_result = trainer.train()
    runtime_s = time.time() - start

    eval_metrics = trainer.evaluate(eval_dataset=eval_tok)
    pred_out = trainer.predict(eval_tok)
    preds = np.argmax(pred_out.predictions, axis=-1)
    labels = pred_out.label_ids

    if preds.shape[0] != labels.shape[0]:
        raise ValueError("Predictions and labels length mismatch.")

    return {
        "trainer": trainer,
        "train_result": train_result,
        "eval_metrics": eval_metrics,
        "preds": preds,
        "labels": labels,
        "runtime_s": runtime_s,
    }

## Experiment A: head-only fine-tuning

This is our baseline adaptation strategy.

Before running, make a hypothesis:
- Will training only the head be enough?
- Which metric (accuracy or macro-F1) do you expect to be stronger/weaker?


In [ ]:
head_model = build_head_only_model()
head_trainable, head_total = count_parameters(head_model)

results_head = train_and_evaluate(
    model=head_model,
    run_name="head_only",
    lr=CONFIG["lr_head"],
    batch_train=CONFIG["train_batch_size"],
    batch_eval=CONFIG["eval_batch_size"],
    epochs=CONFIG["epochs"],
)

print("Head-only eval metrics:")
for k, v in results_head["eval_metrics"].items():
    if isinstance(v, int | float):
        print(f"  {k}: {v:.4f}")

## Experiment B: LoRA fine-tuning

Now we'll run LoRA with the same data split and evaluation protocol.

Focus on three questions:
- Does LoRA improve macro-F1 over head-only?
- How much extra runtime does LoRA require?
- How many trainable parameters does LoRA add relative to head-only?


In [ ]:
lora_model = build_lora_model()
lora_trainable, lora_total = count_parameters(lora_model)

results_lora = train_and_evaluate(
    model=lora_model,
    run_name="lora",
    lr=CONFIG["lr_lora"],
    batch_train=CONFIG["train_batch_size"],
    batch_eval=CONFIG["eval_batch_size"],
    epochs=CONFIG["epochs"],
)

print("LoRA eval metrics:")
for k, v in results_lora["eval_metrics"].items():
    if isinstance(v, int | float):
        print(f"  {k}: {v:.4f}")

## Compare both runs

The table below summarizes efficiency and performance side by side.

Interpretation prompts:
- Which method has the better **macro-F1**?
- Is the performance gain (if any) worth the extra trainable parameters or runtime?
- Do train and eval metrics hint at overfitting?

In applied ML, this tradeoff analysis is often more useful than a single "best" metric.


In [ ]:
comparison = pd.DataFrame(
    [
        {
            "run": "head_only",
            "trainable_params": head_trainable,
            "trainable_percent": 100 * head_trainable / head_total,
            "eval_accuracy": results_head["eval_metrics"].get(
                "eval_accuracy", np.nan
            ),
            "eval_macro_f1": results_head["eval_metrics"].get(
                "eval_macro_f1", np.nan
            ),
            "runtime_s": results_head["runtime_s"],
        },
        {
            "run": "lora",
            "trainable_params": lora_trainable,
            "trainable_percent": 100 * lora_trainable / lora_total,
            "eval_accuracy": results_lora["eval_metrics"].get(
                "eval_accuracy", np.nan
            ),
            "eval_macro_f1": results_lora["eval_metrics"].get(
                "eval_macro_f1", np.nan
            ),
            "runtime_s": results_lora["runtime_s"],
        },
    ]
)
comparison

## Confusion matrices

Confusion matrices reveal *which* classes are confused, not just how many examples are correct.

Use these plots to discuss:
- classes that are consistently difficult,
- whether LoRA helps specific minority classes,
- whether improvements in macro-F1 align with visible per-class gains.


In [ ]:
label_names = None
label_feature = raw["train"].features.get("label", None)
if hasattr(label_feature, "names"):
    label_names = label_feature.names

plot_confusion_matrix(
    results_head["labels"],
    results_head["preds"],
    title="Head-only confusion matrix",
    label_names=label_names,
)

plot_confusion_matrix(
    results_lora["labels"],
    results_lora["preds"],
    title="LoRA confusion matrix",
    label_names=label_names,
)